In [ ]:
import pandas as pd
import glob
import os
import re

def combine_and_prepare():
    # 1. Pfad zu Ihren Dateien
    ordner_pfad = r"C:\Users\flori\Desktop\output_scenarios_test"
    
    # Alle CSV-Dateien suchen
    suchmuster = os.path.join(ordner_pfad, 'output_*.csv')
    files = glob.glob(suchmuster)
    files = [f for f in files if not f.endswith('_ai_ready.csv')]
    
    if not files:
        print(f"Keine passenden Dateien in {ordner_pfad} gefunden.")
        return

    # 2. Dateien nach Setup und Kennlinie (z.B. "A_c" oder "A_w") gruppieren
    # Dateiname ist z.B.: output_1_A_c_20260410.csv -> [output, 1, A, c, 20260410]
    scenarios = {}
    
    for file in files:
        basename = os.path.basename(file)
        parts = basename.replace('.csv', '').split('_')
        
        if len(parts) >= 4:
            day_num = int(parts[1]) # Die Tagesnummer (1 bis 9)
            setup = parts[2]        # 'A'
            kennlinie = parts[3]    # 'c' oder 'w'
            
            group_key = f"{setup}_{kennlinie}"
            
            if group_key not in scenarios:
                scenarios[group_key] = []
            scenarios[group_key].append((day_num, file))

    # 3. Für jede Gruppe (z.B. alle "c" Tage und alle "w" Tage) die Daten zusammenfügen
    for group_key, file_list in scenarios.items():
        # Die Dateien chronologisch nach Tag (1, 2, 3...) sortieren
        file_list.sort(key=lambda x: x[0])
        
        combined_voltages = []
        
        for day_num, file in file_list:
            try:
                df = pd.read_csv(file, sep=';', low_memory=False)
                target_col = 'smartmeter_Spannung U1-N'
                
                if target_col in df.columns:
                    # Spannungswerte des Tages extrahieren
                    voltages = pd.to_numeric(df[target_col], errors='coerce').dropna()
                    combined_voltages.extend(voltages.tolist())
            except Exception as e:
                print(f"Fehler beim Lesen von Tag {day_num} ({file}): {e}")
        
        # 4. Prüfen, ob wir nach dem Zusammenfügen genug Daten für die KI haben
        if len(combined_voltages) >= 672:
            # Wir nehmen exakt 672 Punkte (was genau 7 Tagen entspricht)
            final_data = pd.Series(combined_voltages[:672])
            
            # Speichern als fertige AI-Datei
            out_name = os.path.join(ordner_pfad, f"combined_7days_{group_key}_ai_ready.csv")
            final_data.to_csv(out_name, index=False, header=False)
            
            print(f"[\u2713] Erfolgreich: {len(file_list)} Tage für Szenario '{group_key}' kombiniert.")
            print(f"    Gespeichert als: {os.path.basename(out_name)}")
        else:
            print(f"[!] Warnung: Szenario '{group_key}' hat nach dem Kombinieren aller {len(file_list)} Tage nur {len(combined_voltages)} Messwerte. Die KI benötigt 672.")

if __name__ == "__main__":
    print("Starte das Zusammenfügen der Tages-Dateien...")
    combine_and_prepare()
    print("\nFertig!")